<a href="https://colab.research.google.com/github/brgael/GTD2401-ACD-GA-AD26-07-ml-intermedio/blob/main/04.%20PipeLines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pipelines

En este JNB se revisará el tema de Pipelines.

In [2]:
# Code you have previously used to load data
import pandas as pd
from sklearn.model_selection import train_test_split

# Load data directly from URL
url = 'https://raw.githubusercontent.com/humbertgaar/ml1-unisabana-datasets/main/melb_data.csv'
data = pd.read_csv(url)

# Create target object and call it y
y = data.Price

# Create X
X = data.drop(['Price'], axis=1)

# Split into validation and training data
X_train_full, X_valid_full, y_train, y_valid = train_test_split(
    X, y, train_size=0.8, test_size=0.2, random_state=0
)

# "Cardinality" means the number of unique values in a column
# Select categorical columns with relatively low cardinality
low_cardinality_cols = [
    cname for cname in X_train_full.columns
    if X_train_full[cname].nunique() < 10
    and X_train_full[cname].dtype == "object"
]

# Select numerical columns
numerical_cols = [
    cname for cname in X_train_full.columns
    if X_train_full[cname].dtype in ['int64', 'float64']
]

# Keep selected columns only
my_cols = low_cardinality_cols + numerical_cols

X_train = X_train_full[my_cols].copy()
X_valid = X_valid_full[my_cols].copy()

# Get list of categorical variables
categorical_cols = [
    cname for cname in X_train.columns
    if X_train[cname].dtype == "object"
]

print("Datos cargados correctamente")
print("Tamaño de X_train:", X_train.shape)
print("Tamaño de X_valid:", X_valid.shape)
print("Categorical columns:", categorical_cols)
print("Numerical columns:", numerical_cols)

Datos cargados correctamente
Tamaño de X_train: (10864, 15)
Tamaño de X_valid: (2716, 15)
Categorical columns: ['Type', 'Method', 'Regionname']
Numerical columns: ['Rooms', 'Distance', 'Postcode', 'Bedroom2', 'Bathroom', 'Car', 'Landsize', 'BuildingArea', 'YearBuilt', 'Lattitude', 'Longtitude', 'Propertycount']


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# Preprocessing for numerical data
numerical_transformer = SimpleImputer(strategy='constant')

# Preprocessing for categorical data
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Bundle preprocessing for numerical and categorical data
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

In [4]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=100, random_state=0)

In [5]:
from sklearn.metrics import mean_absolute_error

# Bundle preprocessing and modeling code in a pipeline
my_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', model)
])

# Preprocessing of training data, fit model
my_pipeline.fit(X_train, y_train)

# Preprocessing of validation data, get predictions
preds = my_pipeline.predict(X_valid)

# Evaluate the model
score = mean_absolute_error(y_valid, preds)
print('MAE:', score)

MAE: 160679.18917034855


In [6]:
# Compare actual and predicted values
results = pd.DataFrame({
    'Actual Price': y_valid,
    'Predicted Price': preds
})

print(results.head(10))

       Actual Price  Predicted Price
8505      2165000.0     1.673193e+06
5523       815000.0     8.957633e+05
12852      610000.0     5.726800e+05
4818      1245000.0     1.237080e+06
12812     1160000.0     8.818500e+05
2153      1250000.0     1.364222e+06
9903       715000.0     7.159500e+05
5105       812000.0     9.178300e+05
6107       750500.0     7.669720e+05
8876       610000.0     6.280000e+05
